# Layer 3: two moods that can switch within a session

Layer 2 gave each session one hidden mood for all its tracks: *Engaged* (few skips) or *Browsing* (many skips). It explained most of the differences between sessions, but its predictive checks showed two things it could not reproduce:

- the **within-session gap**: in the real data a skip is followed by another skip more often than the session's own skip rate explains (0.10 in the real data, 0 in Layer 2),
- the **skip/play changes**: real sessions change from skip to play less often than Layer 2 predicts (2.74 against 3.95 per session). Skips come in runs.

In this notebook the mood can **switch** from one track to the next. Each track has its own hidden mood, and the mood of a track only depends on the mood of the track before: this is a hidden Markov model. If the switch probabilities are 0, we are back to Layer 2.

(the goal is to see whether adding switching beats Layer 2)

In [1]:
import json
import os
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mood_switching as ms
import posterior as po
import session_types as st
from checks import (all_checks, average_log_score, load_sessions,
                    real_data_checks, split_users, to_matrix)

SEED = 2026
rng = np.random.default_rng(SEED)

N_DRAWS = 20_000     # importance sampling draws
DF = 4               # degrees of freedom of the Student-t proposal
N_POSTERIOR = 1000   # resampled posterior draws (predictive checks, test score)
N_STARTS = 10        # EM starting points

# Beta(2, 2) for theta_e, theta_b, p0, q_eb and q_be
# EM uses Beta(3, 3) to find the mode on the log-odds scale
PRIOR = ms.PRIOR
EM_PRIOR = po.log_odds_prior(PRIOR)

#make the dirs for the results
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

BLUE = "#00a10d"
ORANGE = "#aa3102"
GREY = "#a4a4a4"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "xtick.color": GREY,
    "ytick.color": GREY,
    "axes.labelcolor": GREY,
})

The model functions are in `mood_switching.py` and the general tools (importance sampling, Hessian, predictive checks) in `posterior.py`, used as they are in Layer 2.
We also import `session_types.py` (Layer 2) to compare the two models.

## 1. The data

We use the same sessions and the same training/test split as Layers 1 and 2 (split by user, seed 2026): 20% of the users are kept aside, only for the additional analysis at the end.

The results of Layers 1 and 2 are read from `results/layer1.json` and `results/layer2.json` for the comparisons.

In [2]:
sessions = load_sessions()
train, test = split_users(sessions, seed=SEED)
train_skips, train_mask = to_matrix(train)
test_skips, test_mask = to_matrix(test)
train_lengths = train_mask.sum(axis=1)

with open("results/layer1.json") as file:
    layer1 = json.load(file)
with open("results/layer2.json") as file:
    layer2 = json.load(file)

print(f"Training: {len(train_skips)} sessions, {train_mask.sum()} tracks, skip rate {train['skip'].mean():.4f}")
print(f"Test: {len(test_skips)} sessions, {test_mask.sum()} tracks")
print(f"Session length: {train_lengths.min()} to {train_lengths.max()} tracks, mean {train_lengths.mean():.2f}")

Training: 4314 sessions, 67122 tracks, skip rate 0.4185
Test: 1061 sessions, 16595 tracks
Session length: 10 to 20 tracks, mean 15.56


We recall the two statistics that Layer 2 could not reproduce, read from `results/layer2.json`:

In [3]:
layer2_checks = pd.DataFrame(layer2["predictive_checks"]).T
layer2_checks.loc[["within-session gap", "skip/play changes"],
                  ["real data", "simulated mean", "share at least as large"]].round(4)

,real data,simulated mean,share at least as large
within-session gap,0.1013,-0.0001,0.0
skip/play changes,2.7415,3.9486,1.0


The training data have 4314 sessions and 67 122 tracks, the same as in Layers 1 and 2 (skip rate around 0.42).
The test data is only used at the end for extra investigation.

The table shows what Layer 3 must fix: Layer 2 gives a within-session gap of 0 (real data 0.10) and too many skip/play changes (3.95 against 2.74).

## 2. The model

For session $i$ with $n_i$ tracks, $y_{it} = 1$ if track $t$ is skipped and $0$ if it is played.

- $z_{it} \in \{E, B\}$ is the hidden mood of track $t$ of session $i$ (Engaged or Browsing). It can now change from one track to the next.
- $p_0 = P(z_{i1} = E)$ is the probability that a session starts Engaged.
- $q_{EB} = P(z_{it} = B \mid z_{i,t-1} = E)$ is the probability to switch from Engaged to Browsing between two tracks, and $q_{BE}$ the probability to switch back.
- $\theta_E$ and $\theta_B$ are the skip rates of the two moods.

**The model**

$$z_{i1} \sim \text{Bernoulli}(p_0), \qquad P(z_{it} = j \mid z_{i,t-1} = k) = A_{kj}, \qquad y_{it} \mid z_{it} \sim \text{Bernoulli}(\theta_{z_{it}})$$

with the transition matrix (rows: mood before, columns: mood after)

$$A = \begin{pmatrix} 1 - q_{EB} & q_{EB} \\ q_{BE} & 1 - q_{BE} \end{pmatrix}$$

The moods form a Markov chain: the mood of a track only depends on the mood of the track before. Given its mood, a track is skipped independently of the others. This is a hidden Markov model.

**Likelihood.** We do not know the moods, so we add all the possible sequences of moods (law of total probability). There are $2^{n_i}$ of them, so we do not write them one by one: the **forward algorithm** adds them one track at a time. With $\alpha_t(j) = P(y_{i1}, \dots, y_{it}, z_{it} = j)$:

$$\alpha_1(j) = P(z_{i1} = j)\, P(y_{i1} \mid j), \qquad \alpha_t(j) = \Big(\sum_k \alpha_{t-1}(k)\, A_{kj}\Big)\, P(y_{it} \mid j), \qquad p(y_i) = \alpha_{n_i}(E) + \alpha_{n_i}(B)$$

These numbers become tiny after many tracks, so the code divides the two $\alpha_t$ by their sum after each track (this is called scaling). The sum is $c_t = P(y_{it} \mid y_{i1}, \dots, y_{i,t-1})$, and $\log p(y_i) = \sum_t \log c_t$. This gives exactly the same numbers as working in logs, but much faster.

**The order of the tracks now matters**: in Layer 2 only $n_i$ and $s_i$ appeared. Here, two skips in a row are more likely than two skips far apart, because the mood tends to stay the same.

**Link with Layer 2**: if $q_{EB} = q_{BE} = 0$, nobody switches, the mood of the first track is the mood of the whole session, and we are back to Layer 2.

**Priors.**

| Parameter | Prior |  |
|---|---|---|
| $\theta_E, \theta_B$ | Beta(2, 2), with $\theta_E < \theta_B$ | Same as Layer 2. The order fixes the labels: "engaged" always means the mood with fewer skips |
| $p_0$ | Beta(2, 2) | Same as Layer 2 |
| $q_{EB}, q_{BE}$ | Beta(2, 2) | We have no strong knowledge about how often listeners switch |

Beta(2, 2) has the weight of 4 observations, while the data have about 63,000 pairs of neighbouring tracks, so the prior should matter very little (we will check it later).

As an example, we take $\theta_E = 0.1$, $\theta_B = 0.7$, $p_0 = 0.5$ and $q_{EB} = q_{BE} = 0.2$, and a session of two skips. By hand, we add the four sequences of moods:

| moods | P(moods) | P(two skips \| moods) | product |
|---|---|---|---|
| E, E | 0.5 × 0.8 = 0.4 | 0.1 × 0.1 = 0.01 | 0.004 |
| E, B | 0.5 × 0.2 = 0.1 | 0.1 × 0.7 = 0.07 | 0.007 |
| B, E | 0.5 × 0.2 = 0.1 | 0.7 × 0.1 = 0.07 | 0.007 |
| B, B | 0.5 × 0.8 = 0.4 | 0.7 × 0.7 = 0.49 | 0.196 |

So $p(y) = 0.214$, and $P(\text{Browsing at track 2} \mid \text{two skips}) = (0.007 + 0.196)/0.214 = 0.949$.

In [ ]:
example = {"theta_e": 0.1, "theta_b": 0.7, "p0": 0.5, "q_eb": 0.2, "q_be": 0.2}
two_skips = np.array([[1, 1]])
two_tracks = np.ones((1, 2), dtype=bool)

likelihood = np.exp(ms.log_likelihood(example, two_skips, two_tracks))
gamma, _ = ms.state_probabilities(example, two_skips, two_tracks)
p_browsing = gamma[0, 1, 1]   # session 0, track 2, Browsing
print(f"Likelihood of two skips: {likelihood:.4f} (by hand 0.214)")
print(f"P(Browsing at track 2 | two skips): {p_browsing:.4f} (by hand 0.949)")

Both numbers agree with the hand computation. With the same skip rates, Layer 2 gave 0.25 and 0.98: here the mood can switch, so two skips say a little less about the mood of each track.

We now check that the order of the tracks matters. The two sessions below both have 2 skips out of 4 tracks, but in the first one the skips are next to each other:

In [ ]:
same_counts = np.array([[1, 1, 0, 0],
                        [1, 0, 1, 0]])
four_tracks = np.ones((1, 4), dtype=bool)
pd.DataFrame({
    "Layer 2": [np.exp(st.log_likelihood(example, row[None], four_tracks)) for row in same_counts],
    "Layer 3": [np.exp(ms.log_likelihood(example, row[None], four_tracks)) for row in same_counts],
}, index=["skip, skip, play, play", "skip, play, skip, play"]).round(4)

Layer 2 gives the same probability to both sessions (only the number of skips counts). Layer 3 finds the first session about 1.6 times more likely, because the skips form a run, which is what the real data showed.

We now want to check that log_likelihood computes the right number, by computing it by brute force and comparing them, as in Layer 2.

We write every possible session of 1 to 4 tracks (30 sessions) in rows of 6 cells, with junk 1s after the end of each session to check that the mask really hides them. For each session we go through **every sequence of moods** ($2^{n_i}$ of them), multiply the probabilities directly (without logs and without the forward algorithm), and add them up.

Then, we check that it agrees with the code, which uses the forward algorithm in logs.

In [ ]:
from itertools import product

# Every session of 1 to 4 tracks:
blocks = []
tiny_lengths = []
for length in range(1, 5):
    codes = np.arange(2 ** length)
    digits = (codes[:, None] >> np.arange(6)) & 1
    # junk 1s after the end of the session
    blocks.append(np.where(np.arange(6) < length, digits, 1))
    tiny_lengths.append(np.full(2 ** length, length))
tiny_skips = np.vstack(blocks)
tiny_lengths = np.concatenate(tiny_lengths)
tiny_mask = np.arange(6) < tiny_lengths[:, None]

q = {"theta_e": 0.23, "theta_b": 0.81, "p0": 0.37, "q_eb": 0.12, "q_be": 0.31}
theta = {"E": q["theta_e"], "B": q["theta_b"]}
start = {"E": q["p0"], "B": 1 - q["p0"]}
switch = {("E", "E"): 1 - q["q_eb"], ("E", "B"): q["q_eb"],
          ("B", "E"): q["q_be"], ("B", "B"): 1 - q["q_be"]}

def brute_force(y):
    '''Add P(moods) * P(y | moods) over every sequence of moods'''
    total = 0
    for moods in product("EB", repeat=len(y)):
        p = start[moods[0]]
        for before, after in zip(moods[:-1], moods[1:]):
            p *= switch[(before, after)]
        for mood, skip in zip(moods, y):
            p *= theta[mood] if skip == 1 else 1 - theta[mood]
        total += p
    return total

direct = np.array([brute_force(row[:length]) for row, length in zip(tiny_skips, tiny_lengths)])
ours = np.array([np.exp(ms.log_likelihood(q, tiny_skips[[i]], tiny_mask[[i]]))
                 for i in range(len(tiny_skips))])
print(f"{len(direct)} sessions, largest difference: {np.abs(ours - direct).max():.1e}")
print(f"Sum over the 16 sessions of 4 tracks: {direct[tiny_lengths == 4].sum():.6f}")

The brute force agrees with the forward algorithm to a rounding error ($10^{-16}$), and the probabilities of all sessions of 4 tracks add up to 1.

We can therefore conclude that the model is a proper probability distribution, that the forward algorithm is right and that the mask works.

Last check: with $q_{EB} = q_{BE} = 0$, Layer 3 should give exactly the Layer 2 log-likelihood. We compare the two on the real training data, with the Layer 2 values of the skip rates and of $p_0$:

In [ ]:
no_switch = {**layer2["em_mode"], "q_eb": 0.0, "q_be": 0.0}
layer2_value = st.log_likelihood(layer2["em_mode"], train_skips, train_mask)
layer3_value = ms.log_likelihood(no_switch, train_skips, train_mask)
print(f"Layer 2: {layer2_value:.6f}")
print(f"Layer 3 without switching: {layer3_value:.6f}")
print(f"Difference: {abs(layer3_value - layer2_value):.1e}")

The two log-likelihoods are equal (up to rounding). So Layer 2 is exactly the special case of Layer 3 with no switching, and any improvement in Layer 3 can only come from the switches.

## 3. Validation on simulated data

Before touching the real data, we fit the model on simulated data, as in Layer 2.

We simulate as many sessions as in the training data, **with the same lengths**.

Our true values are: $\theta_E = 0.08$, $\theta_B = 0.80$, $p_0 = 0.55$, $q_{EB} = 0.05$, $q_{BE} = 0.06$.

- In the long run, the share of Engaged tracks is $q_{BE}/(q_{EB} + q_{BE}) = 0.06/0.11 = 0.55$, the same as $p_0$, so the share of each mood stays about the same along a session.
- The overall skip rate is then about $0.55 \times 0.08 + 0.45 \times 0.80 = 0.40$, close to Yambda's 0.42.
- A mood lasts on average $1/q_{EB} = 20$ tracks (Engaged) and $1/q_{BE} \approx 17$ tracks (Browsing), so a session of 15 tracks has on average about $14.6 \times 0.055 \approx 0.8$ switches: many sessions have none, some have one or two.

We build our simulator with discrete inverse transform through uniforms, as in Layer 2. A number below $p_0$ makes the first track Engaged. Then, at each track, a number below the probability to be Engaged (given the mood before) gives the new mood, and a number below the track's skip rate makes it skipped. We first check it on a large sample of 50 times the training sessions.

In [ ]:
TRUE = {"theta_e": 0.08, "theta_b": 0.80, "p0": 0.55, "q_eb": 0.05, "q_be": 0.06}
sim_skips, sim_mask, sim_engaged = ms.simulate(TRUE, train_lengths, rng)

big_skips, big_mask, big_engaged = ms.simulate(TRUE, np.repeat(train_lengths, 50),
                                               np.random.default_rng(SEED))
again = ms.simulate(TRUE, np.repeat(train_lengths, 50), np.random.default_rng(SEED))
print("Same seed, same output:", np.array_equal(again[0], big_skips))
print("Same lengths as the training data:", np.array_equal(sim_mask, train_mask))

# Pairs of neighbouring tracks: mood before and mood after, where both tracks exist
pairs = big_mask[:, 1:]
before, after = big_engaged[:, :-1][pairs], big_engaged[:, 1:][pairs]

pd.DataFrame({
    "true": [0.55, 0.08, 0.80, 0.05, 0.06, 0.40],
    "simulated": [big_engaged[:, 0].mean(),
                  big_skips[big_engaged & big_mask].mean(),
                  big_skips[~big_engaged & big_mask].mean(),
                  (~after[before]).mean(),     # Engaged then Browsing
                  after[~before].mean(),       # Browsing then Engaged
                  big_skips[big_mask].mean()],
}, index=["first track Engaged (p0)", "skip rate, Engaged", "skip rate, Browsing",
          "switch Engaged to Browsing", "switch Browsing to Engaged",
          "overall skip rate"]).round(4)

Our simulator finds the true values to the 2nd or 3rd decimal.

### 3.1 Finding the mode with EM
EM alternates the same two steps as in Layer 2, but now on tracks instead of sessions (this version of EM for hidden Markov models is called Baum-Welch):

- E-step: for each track, the probability $\gamma_{it}(E)$ that it is Engaged, and for each pair of neighbouring tracks, the probability $\xi_{it}(k, j)$ of mood $k$ then mood $j$. Both come from Bayes' rule, computed with the forward algorithm (section 2) and the same idea run backwards (backward algorithm): $\gamma_{it}(j) = \alpha_t(j)\,\beta_t(j)/p(y_i)$, where $\beta_t(j) = P(y_{i,t+1}, \dots, y_{in_i} \mid z_{it} = j)$.
- M-step: the same Beta mode as in Layer 2, with expected counts:
  - $\theta_E$: expected skips in Engaged tracks over expected Engaged tracks, $\theta_E = (\sum_{i,t} \gamma_{it}(E)\, y_{it} + \alpha - 1)/(\sum_{i,t} \gamma_{it}(E) + \alpha + \beta - 2)$,
  - $p_0$: expected Engaged first tracks over the number of sessions,
  - $q_{EB}$: expected switches from Engaged to Browsing over expected Engaged tracks that have a next track, and the same for $q_{BE}$.

As in Layer 2, EM uses the prior Beta(3, 3), so it finds the peak of the posterior on the log-odds scale (where importance sampling works).

We start EM from 10 random points drawn from the prior. Unlike Layer 2, we do not run a fixed number of iterations: EM stops when one more step improves the log-posterior by less than $10^{-8}$ (at most 2000 steps). When the two moods are hard to tell apart, EM can climb very slowly, and a fixed number of steps could stop before the peak. The last column gives the number of steps each start needed.

We check that the log-posterior never decreases and whether every start ends at the same point. This is more important than in Layer 2: EM for hidden Markov models can stop at a local peak, so different starts could end at different points.

In [ ]:
sim_starts = [ms.random_start(rng) for _ in range(N_STARTS)]
sim_em, sim_mode, sim_histories = po.em_from_starts(ms.em, sim_starts, sim_skips, sim_mask, EM_PRIOR)
sim_em["iterations"] = [len(history) - 1 for history in sim_histories]
sim_em.round({**{name: 6 for name in ms.NAMES}, "log-posterior": 4})

8 of the 10 starts reach the same point in 35 to 81 steps, and the log-posterior never decreases (the largest decrease is 0).

The 2 other starts stop at a local peak where $\theta_E \approx \theta_B \approx 0.41$: the two moods look the same, so EM has no reason to separate them (the bad start described in Layer 2). They never meet the stopping rule (2000 steps), and their log-posterior is much lower (−45,384 against −35,428), so `em_from_starts` keeps the best start. This is why we use several starts.

### 3.2 Importance sampling for the posterior
We build the whole posterior exactly as in Layer 2, with the same functions of `posterior.py`:

- Proposal: a Student-t with 4 degrees of freedom on the log-odds scale, centred at the EM mode, with scale matrix $(-H)^{-1}$, where $H$ is the Hessian of the log-posterior at the mode (central differences). It is now a 5 × 5 matrix.
- Weights $w = \text{posterior}/q$, in logs, normalized with log-sum-exp.
- Draws with $\theta_E \ge \theta_B$ get weight 0.

Each draw needs one forward pass over all sessions, so this step takes about 20 seconds.

In [ ]:
sim_log_post = partial(ms.log_posterior, skips=sim_skips, mask=sim_mask, prior=PRIOR)
sim_fit = po.importance_sampling_at_mode(sim_log_post, po.to_vector(sim_mode, ms.NAMES),
                                         N_DRAWS, rng, DF)
po.posterior_table(sim_fit, ms.NAMES, truth=TRUE).round(5)

All five true values fall within the 95% intervals. The intervals of $q_{EB}$ and $q_{BE}$ are wider (in relative terms) than those of the skip rates: a switch is seen only a few times per session, while every track says something about the skip rates.

### 3.3 Diagnostics of the proposal

We check the same three elements as in Layer 2:
- the number of draws that break θ_E < θ_B (weight 0) (should be low),
- the effective sample size $\text{ESS} = 1/\sum_k \bar w_k^2$ (should be close to the value of a Student-t(4) proposal on a Normal target),
- the largest weight (should be tiny).

In [ ]:
po.proposal_diagnostics(sim_fit).round(6)

The benchmark of Layer 2 (0.866) was computed in 3 dimensions. The ESS of a Student-t proposal depends on the number of dimensions, so we compute it again in **5 dimensions**, with the same functions as in Layer 2:

In [ ]:
# Own generator, so the random draws in the rest of the notebook do not change
check_rng = np.random.default_rng(SEED + 1)
zero, identity = np.zeros(len(ms.NAMES)), np.eye(len(ms.NAMES))
normal_target = partial(po.normal_log_density, center=zero, cov=identity)
t_draws = po.student_t_draws(zero, identity, DF, 100_000, check_rng)
t_check = po.importance_sampling(normal_target, t_draws,
                                 po.student_t_log_density(t_draws, zero, identity, DF))
print(f"ESS share, Student-t(4) proposal for a 5-dimensional Normal target: {t_check['ess_share']:.3f}")

In 5 dimensions the benchmark is 0.80 instead of 0.866: the heavy tails of the Student-t cost more when there are more dimensions. Our ESS share is exactly this value, so the proposal, and so the mode and the Hessian it is built from, fits the posterior. No draw breaks θ_E < θ_B, and the largest weight is tiny.

## 4. Choice of the proposal

As in Layer 2, we compare three proposals on the simulated data set of section 3, with the same number of draws:

- the Student-t (4 df) at the mode (which we have done above),
- the prior itself: Beta(2, 2) draws for each parameter, moved to the log-odds scale,
- a Normal at the mode with half the standard deviation: its tails are lighter than the target's, so it might worsen the weights.

We evaluate and compare our proposals in the same way as in section 3:

In [ ]:
center = sim_fit["center"]
narrow_cov = sim_fit["shape"] / 4   # half the standard deviation
prior_draws = po.prior_draws_log_odds(PRIOR, ms.NAMES, N_DRAWS, rng)
narrow_draws = po.normal_draws(center, narrow_cov, N_DRAWS, rng)
proposals = {
    "Student-t (4 df) at the mode": (
        sim_fit["draws"], po.student_t_log_density(sim_fit["draws"], center, sim_fit["shape"], DF)),
    "prior Beta(2, 2)": (
        prior_draws, po.log_beta_prior_log_odds(prior_draws, PRIOR, ms.NAMES)),
    "Normal at the mode, half the sd": (
        narrow_draws, po.normal_log_density(narrow_draws, center, narrow_cov)),
}

comparison = []
for name, (draws, log_proposal) in proposals.items():
    result = po.importance_sampling(sim_log_post, draws, log_proposal)
    q_eb = po.to_params(draws, ms.NAMES)["q_eb"]
    comparison.append({
        "proposal": name,
        "draws with theta_e < theta_b": result["n_valid"],
        "ESS": round(result["ess"], 1),
        "ESS share": round(result["ess_share"], 5),
        "largest weight": round(result["largest_weight"], 4),
        "q_eb mean": (f"{(result['weights'] * q_eb).sum():.4f}"
                      if result["n_valid"] > 0 else "no valid draw"),
    })
comparison = pd.DataFrame(comparison)
comparison

We observe:

- The Student-t at the mode keeps about 80% of its draws useful, the 5-dimensional benchmark of section 3.3.
- The prior spreads its draws over the whole space, while the posterior only stays within a tiny region. One draw takes all the weight (ESS = 1), so its estimate is just one single draw: its mean of $q_{EB}$ is 0.17, far from the true 0.05.
- The narrow Normal is centred in the right place, so its estimate looks right (0.0507), but its ESS is only about 1.5% of the draws (301 out of 20,000), which signals that the proposal is too narrow. This is even worse than in Layer 2 (6%): with 5 parameters instead of 3, a proposal that is too narrow in every direction misses more of the posterior.

Before we settle on the Student-t, we test its stability, as in Layer 2. We repeat the Student-t run with three other seeds. If the Monte Carlo error is small, the estimates should agree to several decimals.

In [ ]:
stability = []
for seed in [1, 2, 3]:
    fit = po.importance_sampling_at_mode(sim_log_post, center, N_DRAWS,
                                         np.random.default_rng(seed), DF)
    means = po.posterior_table(fit, ms.NAMES)["mean"]
    stability.append({"seed": seed, "ESS share": fit["ess_share"],
                      "largest weight": fit["largest_weight"], **means.to_dict()})
pd.DataFrame(stability).round(5)

The three seeds give the same Student-t estimates to the 3rd or 4th decimal (the largest difference, for $p_0$, is 0.0002, against a posterior standard deviation of 0.009), and the ESS stays near 80%. We thus can infer that the results are stable no matter the seed, and we keep the Student-t.